### Test Envirnment 

- Step 1 → API connection
- Step 2 → PDF text extraction
- Step 3 → Text chunking
- Step 4 → Embeddings API
- Step 5 → FAISS vector index
- Step 6 → Retrieval
- Step 7 → RAG prompt
- Step 8 → LLM answer

#### Step 1 → API connection

In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in .env")


client = InferenceClient(api_key=HF_TOKEN)


In [2]:
# messages = [
#     {   "role" : "user",
#         "content" : "explain what is RAG in one line ?"
#     }
# ]

# response = client.chat_completion(
#     model="MiniMaxAI/MiniMax-M2.5",
#     messages=messages,
#     max_tokens=50
# )

# print(response.choices[0].message.content)


#### Step 2 → PDF text extraction

In [3]:
import fitz #PyMuPdf
import os 

pdf_path = "D:\Code\Python\Intership_Learning\AI_Ml_Projects\RAG_Q&A_Bot\PDFs\Lab-Grown-Food.pdf"

doc = fitz.open(pdf_path) #open in PyMuPdf

pdf_text = ""

for page in doc:
    pdf_text+= page.get_text() + "\n"

page_no = doc.page_count
doc.close()

print(f"PDF Loaded Successfully. \nDocument Total Pages : {page_no}\nNumber of character extracted : {len(pdf_text)}\n")


<>:4: SyntaxWarning: invalid escape sequence '\C'
<>:4: SyntaxWarning: invalid escape sequence '\C'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_20112\3162241734.py:4: SyntaxWarning: invalid escape sequence '\C'
  pdf_path = "D:\Code\Python\Intership_Learning\AI_Ml_Projects\RAG_Q&A_Bot\PDFs\Lab-Grown-Food.pdf"


PDF Loaded Successfully. 
Document Total Pages : 11
Number of character extracted : 4524



#### Note :
- Sometimes PyMuPDF DLL causes import errors on Windows 11. 

- To fix: go to Defender → Virus & Threat Protection → Ransomware Protection → Manage Ransomware Protection → Allow an app through Controlled Folder Access → add your python.exe file.

- Restart VS code editor or Kernal

#### Step 3 → Text chunking

In [4]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
from nltk.tokenize import sent_tokenize

def chunk_text(text, chunk_size=5, overlap=1):
    """
    Splits text into overlapping sentence chunks.

    text: full document string
    chunk_size: number of sentences per chunk
    overlap: number of sentences overlapping between chunks
    """

    sentences = sent_tokenize(text)

    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Create chunks
chunks = chunk_text(pdf_text, chunk_size=5, overlap=1)

print(f"PDF split into {len(chunks)} chunks.")
print(f"\nExample chunk:\n{chunks[0]}")

PDF split into 9 chunks.

Example chunk:
preencoded.png
Lab-Grown Food

preencoded.png
What Is Lab-Grown Food? Lab-grown food, also known as cultivated meat, is made by growing animal 
cells in a laboratory instead of raising or killing animals. These cells are given 
nutrients so they grow into real meat in a controlled environment. 1
2013
The world saw the first lab-
grown burger, a big 
breakthrough in food 
technology. 2
2020
Singapore became the first 
country to allow the sale of 
lab-grown meat.


#### Step 4 → Embeddings API

In [6]:
# Check tourch usage GPU name 
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


#### Main GPU usage → my case NVDIA GPU

In [7]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# Load model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def get_embedding(text):
    # Returns embedding on CPU as numpy array
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    batch_size=32,
    show_progress_bar=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (9, 384)


#### Step 5 → FAISS vector index

In [8]:
import faiss
import numpy as np


faiss.normalize_L2(embeddings) #Normaize embeddings

# Create faiss Index -> faster find similar chunks
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

# Add embeddings to index
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 9 vectors.


In [9]:
# User Question Embeddings :
question = input("Enter Question Here: ....\n")
print(question)
question_embedd = get_embedding(question)

question_embedd = np.array([question_embedd]).astype(np.float32)

faiss.normalize_L2(question_embedd)

What nutrients are given to cells during the growth process according to the document?


In [10]:
# Search FAISS cosign similarity:
top_k = min(5, len(chunks))

distances, indices = index.search(question_embedd, top_k)
print(f"Top Chunk indexes: {indices}")

Top Chunk indexes: [[2 8 0 1 7]]


#### Step 6 → Retrieval

In [11]:
# Retrave Chunk :
retrived_chunk = [chunks[i] for i in indices[0]]

for i, chunk in enumerate(retrived_chunk):
    print(f"\nChunk {i+1}")
    print(chunk)


Chunk 1
(Process)
Cell Collection
A small sample of cells is taken from an 
animal or plant without causing harm. These cells act as the base for producing 
lab-grown food. Cell Growth Process
The cells are placed in a controlled 
laboratory environment and provided with 
nutrients such as amino acids, sugars, and 
vitamins. Harvesting and Processing
Once enough cells are grown, the food 
product is harvested and processed into 
consumable form. •
Although this technology has great potential, its challenges need to 
be solved through more research, investment, and public awareness.

Chunk 2
Lab-grown food shows hope, new ideas, and responsibility.”

preencoded.png
Thank You
We appreciate your time and attention as we explored the future of 
food.

Chunk 3
preencoded.png
Lab-Grown Food

preencoded.png
What Is Lab-Grown Food? Lab-grown food, also known as cultivated meat, is made by growing animal 
cells in a laboratory instead of raising or killing animals. These cells are given 
nutri

#### Step 7 → RAG prompt

In [12]:
context = "\n\n".join(retrived_chunk)

prompt = f"""
You are a strict document QA system.

Rules:
1. Answer ONLY using the provided context.
2. Do NOT use outside knowledge.
3. If the exact answer is not present, reply: "Not found in document".
4. Do not infer or guess.

Context:
{context}

Question:
{question}

Answer:
"""


#### Step 8 → LLM answer

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4o-mini"

client = OpenAI(
    base_url=endpoint,
    api_key=GITHUB_TOKEN
)

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_tokens=200
)

print(response.choices[0].message.content)

Amino acids, sugars, and vitamins.
